#  <center>Taller de Aprendizaje Automatico</center>
##  <center>Paso 3: Procesamiento de datos</center>
### <center>Flujo base para iniciar cualquier estrategia de preprocesamiento en Higgs Boson</center>

Este notebook funciona como nodo base para arrancar cualquier trabajo de procesamiento de datos. La idea es repetir siempre el mismo arranque: cargar los datos de forma reproducible, verificar visualmente que se cargaron bien y dejar separadas las estructuras que despues se usaran para entrenar.

En esta version no se toman todavia decisiones de imputacion, escalado, tratamiento de `-999` ni seleccion avanzada de variables. Solo se deja armado el punto de partida comun.

## Carga reproducible del dataset

Se cargan `training.csv` y `test.csv` desde `data/higgs-boson/`. El archivo de entrenamiento contiene la variable objetivo `Label` y la columna `Weight`; el archivo de test no las incluye.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "roadmap.md").exists() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("No se pudo localizar la raiz del proyecto.")


project_root = find_project_root()
data_dir = project_root / "data" / "higgs-boson"
train_path = data_dir / "training.csv"
test_path = data_dir / "test.csv"

if not train_path.exists():
    raise FileNotFoundError(f"No se encontro el archivo esperado: {train_path}")

if not test_path.exists():
    raise FileNotFoundError(f"No se encontro el archivo esperado: {test_path}")

train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

print(f"project_root = {project_root}")
print(f"train shape = {train_data.shape}")
print(f"test shape = {test_data.shape}")


project_root = /home/santig14/fing/taa/1_TallerAA
train shape = (250000, 33)
test shape = (550000, 31)


## Verificacion visual de la carga

Se muestran las primeras filas de ambos archivos para confirmar rapidamente que la carga fue correcta y recordar la estructura general de cada tabla.

In [2]:
display(train_data.head())
display(test_data.head())


,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt,Weight,Label
0,100000,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,...,2,67.435,2.150,0.444,46.062,1.24,-2.475,113.497,0.002653,s
1,100001,160.937,68.768,103.235,48.146,-999.00,-999.000,-999.000,3.473,2.078,...,1,46.226,0.725,1.158,-999.000,-999.00,-999.000,46.226,2.233584,b
2,100002,-999.000,162.172,125.953,35.635,-999.00,-999.000,-999.000,3.148,9.336,...,1,44.251,2.053,-2.028,-999.000,-999.00,-999.000,44.251,2.347389,b
3,100003,143.905,81.417,80.943,0.414,-999.00,-999.000,-999.000,3.310,0.414,...,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,-0.000,5.446378,b
4,100004,175.864,16.915,134.805,16.405,-999.00,-999.000,-999.000,3.891,16.405,...,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000,6.245333,b


,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_met_phi,PRI_met_sumet,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt
0,350000,-999.000,79.589,23.916,3.036,-999.000,-999.000,-999.000,0.903,3.036,...,2.022,98.556,0,-999.000,-999.000,-999.000,-999.000,-999.000,-999.000,-0.000
1,350001,106.398,67.490,87.949,49.994,-999.000,-999.000,-999.000,2.048,2.679,...,-1.138,176.251,1,47.575,-0.553,-0.849,-999.000,-999.000,-999.000,47.575
2,350002,117.794,56.226,96.358,4.137,-999.000,-999.000,-999.000,2.755,4.137,...,-1.868,111.505,0,-999.000,-999.000,-999.000,-999.000,-999.000,-999.000,0.000
3,350003,135.861,30.604,97.288,9.104,-999.000,-999.000,-999.000,2.811,9.104,...,1.172,164.707,0,-999.000,-999.000,-999.000,-999.000,-999.000,-999.000,0.000
4,350004,74.159,82.772,58.731,89.646,1.347,536.663,-0.339,1.028,77.213,...,-0.231,869.614,3,254.085,-1.013,-0.334,185.857,0.335,2.587,599.213


## Separacion base para entrenamiento

Para definir los inputs del modelo, en esta etapa se excluyen `EventId`, `Weight` y `Label`.

- `EventId` es solo un identificador.
- `Weight` no se usa como feature de entrada.
- `Label` es la variable objetivo y pasa a `y_train`.

Por ahora no se elimina ninguna otra columna. Si mas adelante descartamos alguna feature, esa decision deberia quedar justificada explicitamente.

In [3]:
excluded_train_columns = ["EventId", "Weight", "Label"]
feature_columns = [
    column for column in train_data.columns
    if column not in excluded_train_columns
]

train_event_id = train_data["EventId"].copy()
train_weight = train_data["Weight"].copy()
y_train = train_data["Label"].copy()
X_train = train_data[feature_columns].copy()

test_event_id = test_data["EventId"].copy()
X_test = test_data[feature_columns].copy()

expected_test_columns = [column for column in test_data.columns if column != "EventId"]
if feature_columns != expected_test_columns:
    raise ValueError("Las columnas de entrenamiento y test no coinciden despues de excluir EventId.")

print(f"Cantidad de features para entrenar: {len(feature_columns)}")
print(f"X_train shape = {X_train.shape}")
print(f"y_train shape = {y_train.shape}")
print(f"X_test shape = {X_test.shape}")

column_roles = pd.DataFrame(
    {
        "column": ["EventId", "Weight", "Label"],
        "role": ["identificador", "peso", "target"],
        "used_as_feature": [False, False, False],
    }
)
display(column_roles)


Cantidad de features para entrenar: 30
X_train shape = (250000, 30)
y_train shape = (250000,)
X_test shape = (550000, 30)


,column,role,used_as_feature
0,EventId,identificador,False
1,Weight,peso,False
2,Label,target,False


## Objetos base resultantes

Se muestran las primeras filas de `X_train` y `y_train` para dejar confirmado que la separacion quedo bien armada.

In [4]:
display(X_train.head())
display(y_train.head().to_frame(name="Label"))


,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,...,PRI_met_phi,PRI_met_sumet,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt
0,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,197.760,...,-0.277,258.733,2,67.435,2.150,0.444,46.062,1.24,-2.475,113.497
1,160.937,68.768,103.235,48.146,-999.00,-999.000,-999.000,3.473,2.078,125.157,...,-1.916,164.546,1,46.226,0.725,1.158,-999.000,-999.00,-999.000,46.226
2,-999.000,162.172,125.953,35.635,-999.00,-999.000,-999.000,3.148,9.336,197.814,...,-2.186,260.414,1,44.251,2.053,-2.028,-999.000,-999.00,-999.000,44.251
3,143.905,81.417,80.943,0.414,-999.00,-999.000,-999.000,3.310,0.414,75.968,...,0.060,86.062,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,-0.000
4,175.864,16.915,134.805,16.405,-999.00,-999.000,-999.000,3.891,16.405,57.983,...,-0.871,53.131,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000


,Label
0,s
1,b
2,b
3,b
4,b


## Proximos pasos posibles

- Definir como tratar el marcador `-999`.
- Discutir si conviene conservar o no `Weight` para evaluacion posterior.
- Preparar una primera version del preprocesamiento reutilizable.